In [ ]:
!pip install tensorflow

**🧠 Understanding Step 1: The Dataset**

For a small neural network to learn effectively, it needs a highly cohesive dataset with a restricted vocabulary. By using a focused story about Mars, the ocean, and medieval times, the model can easily pick up on semantic relationships without being overwhelmed by the massive vocabulary of the entire English language.

Note on Results: The code successfully processed our multi-domain story, cleaned the text by removing special characters, and resulted in a raw dataset of 218 total words. This gives us the foundational text to train on.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import re
import time
import sys

# ==========================================
# STEP 1: The Dataset (Expanded for Generalization)
# ==========================================
# We now use a more complex, multi-domain dataset.
# By including Space, Ocean, and Medieval themes, the model must learn
# context separation to avoid generating mixed concepts.

story_text = """
The brave astronauts prepared for their journey to Mars.
The rocket was fueled and ready for launch into the dark sky.
Ten, nine, eight, seven, six, five, four, three, two, one, liftoff!
In space, the stars shone brightly against the black void.
The landing sequence was initiated by the ship computer.
Touchdown was confirmed, and the crew cheered with joy.
Ice was eventually discovered hidden beneath a layer of red dust.

The deep sea submarine descended into the dark oceanic trench.
Water pressure outside the titanium hull was immense and dangerous.
Bioluminescent creatures glowing in the dark swam past the observation window.
The marine biologists documented strange new species of fish and coral.
Sonar systems detected a massive underwater cave system ahead.
They carefully navigated the submarine through the narrow rocky passages.
Finding a hydrothermal vent teeming with life was their greatest discovery.

The ancient knights rode their armored horses across the green valley.
Swords clashed and shields shattered during the fierce medieval tournament.
The king watched proudly from his high stone castle balcony.
Archers readied their bows, aiming arrows at the wooden targets.
A fierce dragon was rumored to sleep inside the mountain caves.
Villagers gathered in the market to trade bread, armor, and horses.
The brave knights swore an oath to protect the peaceful kingdom forever.
"""

# Clean the text: lowercase it and remove special characters
clean_story = re.sub(r'[^a-z0-9\s]', '', story_text.lower())
words = clean_story.split()

print(f"Dataset ready. words: {words}")
print(f"Dataset ready. Total words: {len(words)}")

Dataset ready. words: ['the', 'brave', 'astronauts', 'prepared', 'for', 'their', 'journey', 'to', 'mars', 'the', 'rocket', 'was', 'fueled', 'and', 'ready', 'for', 'launch', 'into', 'the', 'dark', 'sky', 'ten', 'nine', 'eight', 'seven', 'six', 'five', 'four', 'three', 'two', 'one', 'liftoff', 'in', 'space', 'the', 'stars', 'shone', 'brightly', 'against', 'the', 'black', 'void', 'the', 'landing', 'sequence', 'was', 'initiated', 'by', 'the', 'ship', 'computer', 'touchdown', 'was', 'confirmed', 'and', 'the', 'crew', 'cheered', 'with', 'joy', 'ice', 'was', 'eventually', 'discovered', 'hidden', 'beneath', 'a', 'layer', 'of', 'red', 'dust', 'the', 'deep', 'sea', 'submarine', 'descended', 'into', 'the', 'dark', 'oceanic', 'trench', 'water', 'pressure', 'outside', 'the', 'titanium', 'hull', 'was', 'immense', 'and', 'dangerous', 'bioluminescent', 'creatures', 'glowing', 'in', 'the', 'dark', 'swam', 'past', 'the', 'observation', 'window', 'the', 'marine', 'biologists', 'documented', 'strange', 'n

**🔢 Understanding Step 2: Tokenization**

Language models process mathematics, not text. Tokenization is the bridge between human language and machine understanding.

Vocabulary (vocab_size): The total number of unique words our model knows.
Padding (<PAD>): A special token used to ensure all our input sequences are the same length, which is a requirement for efficient matrix math in GPUs.

In [ ]:
# ==========================================
# STEP 2: The Tokenizer (Words → Numbers)
# ==========================================
# Neural networks only understand numbers, so we map every word to an integer ID.
unique_words = sorted(list(set(words)))

# Create word-to-index and index-to-word mappings
word2idx = { word:i+1 for i,word in enumerate(unique_words)}
idx2word = { i:word for word,i in word2idx.items()}

# Add a special <PAD> token for empty spaces
word2idx['<PAD>'] = 0
idx2word[0] = '<PAD>'

VOCAB_SIZE = len(word2idx)
print(VOCAB_SIZE)

# Encode the entire story into a sequence of integer IDs
encoded_story = [word2idx[w] for w in words]
print(f"Vocabulary Size: {VOCAB_SIZE}")
print(f"First 5 words encoded: {encoded_story[:5]}")
print(f"Length of story: {len(encoded_story)}")

162
Vocabulary Size: 162
First 5 words encoded: [139, 21, 13, 103, 53]
Length of story: 218


Note on Results: The tokenizer extracted the unique words and built our vocabulary. We now have a Vocabulary Size of 162 tokens (including the padding token). It also successfully translated the first 5 words into their new integer IDs.

**🪟 Understanding Step 3: The Sliding Window (Causal Language Modeling)**
GPT models are trained on a simple objective: Predict the next word. We create our training data by sliding a 'window' of SEQ_LENGTH (5 words) across our text. For example, if the text is [A, B, C, D, E, F]:

Input: [A, B, C, D, E] -> Target: [F]
This forces the model to learn context and grammar to accurately guess what comes next.

In [ ]:
# ==========================================
# STEP 3: Creating Training Sequences
# ==========================================
# We slide a window across the text to teach the model to predict the next word.
SEQ_LENGTH = 5  # The model will look at 5 words to predict the 6th

input_sequences = []
target_words = []

for i in range(len(encoded_story) - SEQ_LENGTH):
  # Extract a sequence of 5 words
  seq = encoded_story[i: i + SEQ_LENGTH]
  # The target is the very next word
  target = encoded_story[i + SEQ_LENGTH]

  input_sequences.append(seq)
  target_words.append(target)

X = np.array(input_sequences)
y = np.array(target_words)

print(f"Generated {len(X)} training sequences.")
print(f"Input sequence:{X}")
print(f"Ouput sequence:{y}")


Generated 213 training sequences.
Input sequence:[[139  21  13 103  53]
 [ 21  13 103  53 140]
 [ 13 103  53 140  74]
 ...
 [  6  94 145 105 139]
 [ 94 145 105 139 102]
 [145 105 139 102  77]]
Ouput sequence:[140  74 145  86 139 110 156  57   8 108  53  80  73 139  37 122 138  93
  47 116 121  52  55 142 150  98  83  70 125 139 127 120  23   3 139  19
 155 139  79 115 156  71  24 139 119  31 146 156  32   8 139  35  29 160
  75  68 156  48  41  62  16   1  81  97 109  46 139  38 114 130  39  73
 139  37  96 149 158 104  99 139 144  66 156  69   8  36  18  34  59  70
 139  37 131 101 139  95 159 139  84  17  43 129  92 126  97  51   8  33
 124 135  40   1  87 151  27 134   4 141  25  91 139 130 143 139  90 111
 100  50   1  67 153 137 160  82 156 140  60  42 139   7  78 112 140  11
  65   2 139  61 152 132  30   8 118 117  45 139  49  88 147 139  76 157
 106  56  64  63 128  26  15   9 107 140  20   5  12  14 139 161 136   1
  49  44 156 113 145 123  72 139  89  28 154  58  70 139  85 1

Note on Results: By sliding the 5-word window across our encoded story, the code generated 213 distinct training sequences. These X (inputs) and y (targets) arrays are what the neural network will actually learn from.

In [ ]:
# ==========================================
# STEP 4: The Core GPT Architecture
# ==========================================
# Building a miniature Transformer model for language modeling.
def build_simple_gpt(vocab_size, seq_length):
    inputs = keras.Input(shape=(seq_length,))

    # 1. Embeddings: Convert word IDs to dense vectors
    token_embedding = layers.Embedding(input_dim=vocab_size, output_dim=64)(inputs)

    # 2. Positional Encoding: Give the model a sense of word order
    positions = tf.range(start=0, limit=seq_length, delta=1)
    pos_embedding = layers.Embedding(input_dim=seq_length, output_dim=64)(positions)
    x = token_embedding + pos_embedding

    # 3. Multi-Head Attention: Allow the model to look at the context of the sentence
    # (Using a causal mask implicitly by processing sequences left-to-right)
    # use_causal_mask=True makes this a real GPT — each word can only see previous words, not future ones
    attention_output = layers.MultiHeadAttention(num_heads=4, key_dim=16)(x, x, use_causal_mask=True)
    x = layers.LayerNormalization()(x + attention_output)

    # 4. Feed Forward Network: Process the attended representations
    ffn_output = layers.Dense(128, activation='relu')(x)
    ffn_output = layers.Dense(64)(ffn_output)
    x = layers.LayerNormalization()(x + ffn_output)

    # 5. Pooling: Flatten the sequence before the final prediction layer
    x = layers.GlobalAveragePooling1D()(x)

    # 6. Output Layer: Predict the probability of every word in the vocabulary
    outputs = layers.Dense(vocab_size, activation='softmax')(x)

    model = keras.Model(inputs=inputs, outputs=outputs)
    model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

gpt_model = build_simple_gpt(VOCAB_SIZE, SEQ_LENGTH)
print("GPT Architecture built successfully.")

| Head | Learns            |
| ---- | ----------------- |
| 1    | grammar           |
| 2    | subject/object    |
| 3    | long dependencies |
| 4    | semantic meaning  |


In [ ]:
# ==========================================
# STEP 4: The Core GPT Architecture
# ==========================================
# Building a miniature Transformer model for language modeling.

def build_simple_gpt(vocab_size, seq_length):
  # Creates input placeholder.input looks like: [12, 45, 87, 3, 91]
  inputs = keras.Input(shape=(seq_length,))

  # 1. Embeddings: Convert word IDs to dense vectors
  token_embedding = layers.Embedding(input_dim=vocab_size, output_dim=64)(inputs)

  # 2. Positional Encoding: Give the model a sense of word order
  positions = tf.range(start=0, limit=seq_length, delta=1)
  positional_encoding = layers.Embedding(input_dim=seq_length, output_dim=64)(positions)
  x = token_embedding + positional_encoding

  # 3. Multi-Head Attention: Allow the model to look at the context of the sentence
  # (Using a causal mask implicitly by processing sequences left-to-right)
  # use_causal_mask=True makes this a real GPT — each word can only see previous words, not future ones
  # key_dim=16 -> 16-dimensional Q/K/V vectors -> 4 heads × 16 = 64
  attention_output = layers.MultiHeadAttention(num_heads=4, key_dim=16)(x, x, use_causal_mask=True)
  x = layers.LayerNormalization()(x + attention_output)

  # 4. Feed Forward Network: Process the attended representations
  ffn_ouput = layers.Dense(128, activation='relu')(x)
  ffn_ouput = layers.Dense(64)(ffn_ouput)
  x = layers.LayerNormalization()(x + ffn_ouput)

  # 5. Pooling: Flatten the sequence before the final prediction layer
  x = layers.GlobalAveragePooling1D()(x)

  # 6. Output Layer: Predict the probability of every word in the vocabulary
  outputs = layers.Dense(vocab_size, activation='softmax')(x)

  model = keras.Model(inputs=inputs, outputs=outputs)
  model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
  return model

gpt_model = build_simple_gpt(VOCAB_SIZE, SEQ_LENGTH)
print("GPT Architecture built successfully.")


GPT Architecture built successfully.


In [ ]:
gpt_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 5)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 5, 64)     │     10,368 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 5, 64)     │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 5, 64)     │     16,640 │ add[0][0],        │
│ (MultiHeadAttentio… │                   │            │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 5, 64)     │          0 │ add[0][0],        │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 5, 64)     │        128 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 5, 128)    │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 5, 64)     │      8,256 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 5, 64)     │          0 │ layer_normalizat… │
│                     │                   │            │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 5, 64)     │        128 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 162)       │     10,530 │ global_average_p… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 54,370 (212.38 KB)

 Trainable params: 54,370 (212.38 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# ==========================================
# STEP 5: Train the Model (Pre-training)
# ==========================================
# The model learns the statistical patterns of the story.
print("Starting Pre-training (Unsupervised Next-Word Prediction)...")

# Train for 50 epochs to ensure the model captures the patterns well.
history = gpt_model.fit(X, y, epochs=50, batch_size=8, verbose=1)
print("Pre-training Complete!")

Starting Pre-training (Unsupervised Next-Word Prediction)...
Epoch 1/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 9s 20ms/step - accuracy: 0.0563 - loss: 5.1375
Epoch 2/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.1174 - loss: 4.5417
Epoch 3/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.1737 - loss: 4.1566
Epoch 4/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.2113 - loss: 3.7604
Epoch 5/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.2911 - loss: 3.3960
Epoch 6/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.3568 - loss: 3.0298
Epoch 7/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.4977 - loss: 2.6707
Epoch 8/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5681 - loss: 2.3096
Epoch 9/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.6854 - loss: 1.9909
Epoch 10/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.8075 - loss: 1.6574
Epoch 11/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8779 - loss: 1.3284
Epoc

Input prompt
    ->
Predict next word
    ->
Append predicted word
    ->
Use updated sentence again
    ->
Predict next word
    ->
Repeat

"The brave astronauts"

↓

"The brave astronauts explored"

↓

"The brave astronauts explored the"

↓

"The brave astronauts explored the mysterious"

...

In [ ]:
# ==========================================
# STEP 6: Text Generation
# ==========================================
# Using the trained model to generate new text autoregressively with temperature sampling.
def generate_text(model, prompt_text, num_words=15, temperature=0.8):
    print(f"\n--- Generating Text (Temp: {temperature}) ---")
    print(f"Prompt: '{prompt_text}'\n")

    clean_prompt = re.sub(r'[^a-z0-9\s]', '', prompt_text.lower())
    current_words = clean_prompt.split()

    sys.stdout.write(prompt_text + " ")
    sys.stdout.flush()

    for _ in range(num_words):
        # Grab the last words to act as context
        window = current_words[-SEQ_LENGTH:]
        window_ids = [word2idx.get(w, 0) for w in window]

        # Pad the window if it's shorter than the required SEQ_LENGTH
        if len(window_ids) < SEQ_LENGTH:
            window_ids = [0] * (SEQ_LENGTH - len(window_ids)) + window_ids

        input_tensor = np.array([window_ids])
        predictions = model.predict(input_tensor, verbose=0)[0]

        # Temperature scaling for creativity
        if temperature <= 0:
            next_word_id = np.argmax(predictions)
        else:
            predictions = np.log(predictions + 1e-7) / temperature
            exp_preds = np.exp(predictions)
            predictions = exp_preds / np.sum(exp_preds)
            next_word_id = np.random.choice(len(predictions), p=predictions)

        next_word = idx2word.get(next_word_id, "<UNK>")

        sys.stdout.write(next_word + " ")
        sys.stdout.flush()
        time.sleep(0.1)

        current_words.append(next_word)

    print("\n\n--- Generation Complete ---")

# Test generation
generate_text(gpt_model, "The brave astronauts", num_words=15, temperature=0.7)


--- Generating Text (Temp: 0.7) ---
Prompt: 'The brave astronauts'

The brave astronauts readied journey observation journey to of was documented was fueled sea launch fish an into 

--- Generation Complete ---


In [ ]:
# ==========================================
# STEP 7: Supervised Fine-Tuning
# ==========================================
# Adapting the pre-trained language model to perform sentiment classification.

# 1. Dataset for Fine-Tuning (1 = Positive, 0 = Negative)
# Expanded to 20 examples so the model has enough signal to learn the task.
# Rule of thumb: fine-tuning needs at least 10-20 examples per class.
fine_tune_data = [
    # --- Positive examples (success, discovery, joy) ---
    ("The astronauts successfully reached Mars", 1),
    ("Touchdown was confirmed and the crew cheered", 1),
    ("The mission of exploration would truly begin", 1),
    ("The rocket soared into the dark sky", 1),
    ("Ice was discovered hidden beneath red dust", 1),
    ("Finding a hydrothermal vent teeming with life", 1),
    ("The brave knights protected the peaceful kingdom", 1),
    ("The king watched proudly from his high castle", 1),
    ("The marine biologists documented strange new species", 1),
    ("The crew celebrated their greatest discovery", 1),
    # --- Negative examples (failure, danger, disaster) ---
    ("The ship systems failed during the descent", 0),
    ("The computer crashed and they lost control", 0),
    ("The mission was a disaster", 0),
    ("They ran out of food and oxygen", 0),
    ("Water pressure outside the hull was immense and dangerous", 0),
    ("Swords clashed and shields shattered during the fierce tournament", 0),
    ("A fierce dragon threatened to destroy the village", 0),
    ("The submarine lost power in the dark trench", 0),
    ("The rocket engine failed on the launch pad", 0),
    ("The crew was stranded with no way to return", 0),
]

ft_sequences, ft_labels = [], []
for text, label in fine_tune_data:
    clean_text = re.sub(r'[^a-z0-9\s]', '', text.lower())
    seq = [word2idx.get(w, 0) for w in clean_text.split()]
    # Pad or truncate
    seq = seq[:SEQ_LENGTH] if len(seq) > SEQ_LENGTH else [0] * (SEQ_LENGTH - len(seq)) + seq
    ft_sequences.append(seq)
    ft_labels.append(label)

X_ft = np.array(ft_sequences)
y_ft = np.array(ft_labels)

# 2. Build Fine-Tuned Model Architecture
# Unfreeze the base model to allow the embeddings to adjust slightly
gpt_model.trainable = True

# Extract the pooled output from the second-to-last layer of the pre-trained model
pooled_output = gpt_model.layers[-2].output

# Add a fresh classification head
ft_outputs = layers.Dense(1, activation='sigmoid', name='sentiment_classifier')(pooled_output)
finetuned_classifier = keras.Model(inputs=gpt_model.input, outputs=ft_outputs)

finetuned_classifier.compile(
    loss='binary_crossentropy',
    optimizer=keras.optimizers.Adam(learning_rate=5e-5),
    metrics=['accuracy']
)

print("Starting Supervised Fine-Tuning...")
finetuned_classifier.fit(X_ft, y_ft, epochs=50, verbose=0)
print("Fine-tuning Complete!")

# 3. Final Test of the Classifier
test_sentences = ["The crew cheered with joy", "The system failed completely"]
print("\n--- Fine-Tuned Classifier Results ---")
for text in test_sentences:
    clean_text = re.sub(r'[^a-z0-9\s]', '', text.lower())
    seq = [word2idx.get(w, 0) for w in clean_text.split()]
    seq = seq[:SEQ_LENGTH] if len(seq) > SEQ_LENGTH else [0] * (SEQ_LENGTH - len(seq)) + seq

    prediction = finetuned_classifier.predict(np.array([seq]), verbose=0)[0][0]
    sentiment = "Positive 🟢" if prediction > 0.5 else "Negative 🔴"
    print(f"Sentence: '{text}' -> {sentiment} (Score: {prediction:.3f})")


Starting Supervised Fine-Tuning...
Fine-tuning Complete!

--- Fine-Tuned Classifier Results ---
Sentence: 'The crew cheered with joy' -> Negative 🔴 (Score: 0.320)
Sentence: 'The system failed completely' -> Negative 🔴 (Score: 0.257)
